# Quadruped Locomotion (Go2): 보상 단계별 이어학습

전체 학습(2,000,000 step)은 Colab 에서 끝까지 돌리기 어렵습니다. 그래서 각 보상 단계로
**거의 끝까지(~2,000,000 step) 미리 학습해 둔 체크포인트**를 불러와, Colab 에서는
**1,000 step 만 이어서** 학습하며 결과를 확인합니다.

이 노트북은 보상 설정이 단계적으로 어떻게 바뀌는지 설명하고, 각 단계 모델을 이어학습합니다.

- **라운드 1** : `env1` (기본 task-only 보상) → `pretrained_env1` 를 `envs1.yaml` 로 1,000 step 이어학습
- **라운드 2** : `env1 → env2` 보상 변화 설명 → `pretrained_env2` 를 `envs2.yaml` 로 1,000 step 이어학습
- **라운드 3** : `env2 → env3` 보상 변화 설명 → `pretrained_env3` 를 `envs3.yaml` 로 1,000 step 이어학습

> 런타임 → 런타임 유형 변경 → **T4 GPU** 로 설정 후 실행하세요.


---

## 0. 환경 설정

GitHub 레포지토리를 clone 하고 의존성을 설치합니다. (이전 노트북들과 동일)


In [ ]:
# Clone repository
import os, sys

import yaml

# Detect Colab by availability of /content or google.colab.
try:
    import google.colab  # noqa: F401
    in_colab = True
except Exception:
    in_colab = os.path.isdir("/content")

try:
    base_dir
except NameError:
    base_dir = "/content" if in_colab else os.getcwd()
os.chdir(base_dir)

repo_dir = os.path.join(base_dir, "RL_tutorial")

print(f"Base directory: {base_dir}")
print(f"Repo directory: {repo_dir}")

if not os.path.isdir(repo_dir):
  !git clone https://github.com/agbread/RL_tutorial.git
else:
  print("Cloned Directory already exists")

os.chdir(repo_dir)
print("Current Directory: ", os.getcwd())

sys.path.insert(0, os.path.join(repo_dir, "src"))
os.environ["MUJOCO_GL"] = "egl"

# numpy 1.x / 2.x 호환성 패치:
# 저장된 모델이 numpy 2.x (numpy.core_ 경로) 로 직렬화된 경우를 위해
# numpy.core_ 를 numpy.core 의 alias 로 등록합니다.
import numpy, numpy.core, numpy.core.numeric, numpy.core.multiarray
import numpy.random._pickle as _np_pickle

sys.modules['numpy.core_'] = numpy.core
sys.modules['numpy.core_.numeric'] = numpy.core.numeric
sys.modules['numpy.core_.multiarray'] = numpy.core.multiarray

_orig_bg_ctor = _np_pickle.__bit_generator_ctor
def _patched_bg_ctor(bg='MT19937'):
    return bg() if isinstance(bg, type) else _orig_bg_ctor(bg)
_np_pickle.__bit_generator_ctor = _patched_bg_ctor

In [ ]:
# Install dependencies
# stable-baselines3는 PyPI에서 설치합니다 (이 repo에는 sb3 소스 포크가 없음).
# 로컬에서 검증된 버전 조합으로 고정합니다.
!pip install "stable-baselines3==2.3.0" "gymnasium==0.29.1" "mujoco==3.8.0" "numpy<2" "imageio[ffmpeg]" tensorboard pygments

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter
import inspect

def _render_code(code, title="code", max_height=400, bg="transparent", indent=16):
    style_name = "native" if in_colab else "friendly"
    formatter = HtmlFormatter(style=style_name, noclasses=True, linenos="inline")
    html = highlight(code, PythonLexer(), formatter)
    css = """
    <style>
    .highlight pre { margin: 0; text-align: left; }
    </style>
    """
    return HTML(f"""
    {css}
    <details>
      <summary>{title}</summary>
      <div style="margin-top:8px; margin-left:{indent}px; max-height:{max_height}px; overflow:auto; border:1px solid #ddd; padding:10px; background:{bg};">
        {html}
      </div>
    </details>
    """)


def show_code(path, max_height=400, bg="transparent"):
    code = Path(path).read_text()
    return _render_code(code, title=str(path), max_height=max_height, bg=bg)

def show_func(obj, max_height=400, bg="transparent"):
    code = inspect.getsource(obj)
    return _render_code(code, max_height=max_height, bg=bg)

---

## 1. 설정 파일 살펴보기

- **`src/params.yaml`** — PPO 하이퍼파라미터/학습 설정
- **`src/mdp/reward.py`** — 보상 함수 구현

보상 가중치(`envsN.yaml`)는 각 라운드 상단에서 해당 단계의 것만 펼쳐 봅니다.


In [ ]:
display(show_code(f"{repo_dir}/src/params.yaml"))
display(show_code(f"{repo_dir}/src/mdp/reward.py", max_height=600))

---

## 2. 공통 헬퍼 정의

세 라운드에서 반복 사용할 함수와 설정을 정의합니다.

- 상단 설정: `ROUND1_MODEL`/`ROUND1_CFG` ~ `ROUND3_MODEL`/`ROUND3_CFG`, `ADDITIONAL_TIMESTEPS`
- `reward_diff(a, b)` — 두 envs.yaml 사이에 바뀐 보상/설정 항목을 출력
- `rollout_and_video(model, cfg, tag)` — 모델을 롤아웃해 mp4 저장 (test 로직 이식)
- `show_video(path)` — 노트북에서 영상 재생
- `finetune(model_in, cfg, tag)` — 해당 보상 설정으로 1,000 step 이어학습


In [ ]:
import time, gc, shutil
import numpy as np
import imageio
from tqdm.auto import tqdm

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, CallbackList,
)
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv

import src.go2_mujoco_env as go2_env
from src.utils.reward_logging_callback import RewardLoggingCallback
from IPython.display import Video

# 학습 설정 로드
with open(f"{repo_dir}/src/params.yaml", "r", encoding="utf-8") as f:
    policy_cfg = yaml.safe_load(f)

# ===== 설정 (필요시 수정) =====
# 각 라운드는 자기 사전학습 모델에서 독립적으로 출발한다.
ROUND1_MODEL = f"{repo_dir}/models/pretrained_env1/best_model.zip"
ROUND1_CFG   = f"{repo_dir}/src/envs1.yaml"
ROUND2_MODEL = f"{repo_dir}/models/pretrained_env2/best_model.zip"
ROUND2_CFG   = f"{repo_dir}/src/envs2.yaml"
ROUND3_MODEL = f"{repo_dir}/models/pretrained_env3/best_model.zip"
ROUND3_CFG   = f"{repo_dir}/src/envs3.yaml"
ADDITIONAL_TIMESTEPS = 1_000   # 매 라운드 추가 학습 step
N_ENVS = policy_cfg["n_envs"]   # Colab 자원에 맞게 줄여도 됨 (예: 4)
SEED = policy_cfg["seed"]

for _m in (ROUND1_MODEL, ROUND2_MODEL, ROUND3_MODEL):
    assert os.path.exists(_m), f"모델을 찾을 수 없습니다: {_m}"

VIDEO_DIR = f"{repo_dir}/models/_reward_tuning_videos"
os.makedirs(VIDEO_DIR, exist_ok=True)


def rollout_and_video(model_path, env_cfg_path, tag):
    """모델을 롤아웃하여 mp4 로 저장하고 경로를 반환."""
    env = go2_env.Go2MujocoEnv(
        prj_path=repo_dir, cfg_path=env_cfg_path,
        given_command=[0.9, 0.0, 0.0],
        render_mode="rgb_array", camera_name="tracking",
        width=960, height=540,
    )
    env._reset_noise_scale = 0.05
    custom_objects = {
        "observation_space": env.observation_space,
        "action_space": env.action_space,
    }
    model = PPO.load(model_path, env=env, custom_objects=custom_objects, verbose=0)

    video_fps = 10
    render_interval = 50 // video_fps               # control 50Hz
    max_steps = int(policy_cfg["test"]["max_time_step_s"] * 50)
    video_path = f"{VIDEO_DIR}/rollout_{tag}.mp4"

    obs, _ = env.reset()
    frames = []
    for step in tqdm(range(max_steps), desc=f"rollout[{tag}]", unit="step"):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        if step % render_interval == 0:
            frames.append(env.render())
        if terminated or truncated:
            obs, _ = env.reset()
    env.close()

    imageio.mimwrite(video_path, frames, fps=video_fps,
                     codec="libx264", quality=8, pixelformat="yuv420p")
    print("saved video:", video_path)
    return video_path


def show_video(video_path):
    display(Video(video_path, embed=True, html_attributes="controls autoplay loop"))



def finetune(model_in_path, env_cfg_path, run_tag):
    """env_cfg_path 보상으로 model_in_path 를 ADDITIONAL_TIMESTEPS 만큼 이어학습."""
    log_dir = f"{repo_dir}/logs"
    os.makedirs(log_dir, exist_ok=True)
    env_kwargs = {"prj_path": repo_dir, "cfg_path": env_cfg_path}
    vec_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                           n_envs=N_ENVS, seed=SEED, vec_env_cls=SubprocVecEnv)
    eval_env = make_vec_env(go2_env.Go2MujocoEnv, env_kwargs=env_kwargs,
                            n_envs=1, seed=SEED + 10_000, vec_env_cls=DummyVecEnv)

    run_name = time.strftime("%Y-%m-%d_%H-%M-%S") + f"-{run_tag}"
    model_path = f"{repo_dir}/models/{run_name}"
    os.makedirs(model_path, exist_ok=True)
    shutil.copy2(env_cfg_path, f"{model_path}/envs.yaml")   # 사용한 보상 설정 보관
    print("저장 위치:", model_path)

    _dummy = go2_env.Go2MujocoEnv(prj_path=repo_dir, cfg_path=env_cfg_path, render_mode=None)
    custom_objects = {
        "observation_space": _dummy.observation_space,
        "action_space": _dummy.action_space,
    }
    _dummy.close()

    callbacks = CallbackList([
        EvalCallback(eval_env, best_model_save_path=model_path, log_path=log_dir,
                     eval_freq=max(policy_cfg["eval_freq"] // N_ENVS, 1),
                     n_eval_episodes=5, deterministic=True, render=False),
        CheckpointCallback(
            save_freq=max(policy_cfg["policy"]["n_steps"] * policy_cfg["log"]["interval"] // N_ENVS, 1),
            save_path=model_path, name_prefix="model",
            save_replay_buffer=False, save_vecnormalize=False),
        RewardLoggingCallback(),
    ])

    print(f"[{run_tag}] load pretrained: {model_in_path}")
    model = PPO.load(model_in_path, env=vec_env, custom_objects=custom_objects,
                     verbose=1, tensorboard_log=log_dir)
    model.learning_rate = policy_cfg["policy"]["learning_rate"]
    model._setup_lr_schedule()
    model.learn(total_timesteps=ADDITIONAL_TIMESTEPS, reset_num_timesteps=False,
                progress_bar=True, tb_log_name=run_name, callback=callbacks)
    model.save(f"{model_path}/final_model")

    vec_env.close(); eval_env.close()
    del model; gc.collect()

    out = f"{model_path}/best_model.zip"
    if not os.path.exists(out):
        out = f"{model_path}/final_model.zip"
    print(f"[{run_tag}] saved ->", out)
    return out


def reward_diff(cfg_a_path, cfg_b_path):
    """두 envs.yaml 사이에 바뀐 항목(cost/reward/termination 등)을 출력."""
    a = yaml.safe_load(open(cfg_a_path, encoding="utf-8"))
    b = yaml.safe_load(open(cfg_b_path, encoding="utf-8"))
    rows = []
    def walk(da, db, prefix=""):
        for k in dict.fromkeys(list(da) + list(db)):
            va, vb = da.get(k), db.get(k)
            if isinstance(va, dict) or isinstance(vb, dict):
                walk(va or {}, vb or {}, prefix + k + ".")
            elif va != vb:
                rows.append((prefix + k, va, vb))
    walk(a, b)
    print(f"보상/설정 변화: {os.path.basename(cfg_a_path)} -> {os.path.basename(cfg_b_path)}")
    for key, va, vb in rows:
        print(f"  {key:26s}: {va}  ->  {vb}")
    if not rows:
        print("  (변경 없음)")
    return rows


# 비교용 영상 목록
round_videos = []

---

## 3. 라운드 1 — env1 (기본 task-only 보상)

`env1` 은 추가 보상 없이 **속도 추종(`linear/angular_vel_tracking`)만** 보는 가장 기본적인
*task-only* 설정입니다. `env1` 보상으로 ~2,000,000 step 미리 학습해 둔 체크포인트
(`pretrained_env1`)를 Colab 에서는 **1,000 step 만 이어서** 학습합니다.

### (a) 보상 설정 확인 (envs1)

env1 은 정규화 패널티·걸음새 보상·생존 보상이 모두 0 이고 `allow_calf_contact` 도 켜져 있어
(정강이 접촉 허용) 제약이 가장 적습니다. 이후 라운드에서 여기에 보상을 더해갑니다.


In [ ]:
display(show_code(f"{repo_dir}/src/envs1.yaml"))

### (b) env1 모델 이어학습 후 확인

`pretrained_env1` 체크포인트를 **`envs1.yaml` 보상으로 1,000 step 이어학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


In [ ]:
r1_model = finetune(ROUND1_MODEL, ROUND1_CFG, run_tag="env1")
vp = rollout_and_video(r1_model, ROUND1_CFG, tag="env1_after")
round_videos.append((vp, "env1 (1,000 step 이어학습 후)"))
show_video(vp)

---

## 4. 라운드 2 — env1 → env2 (정규화·걸음새 보상 도입)

`env2` 보상으로 **거의 끝까지(~2,000,000 step) 미리 학습해 둔 체크포인트**(`pretrained_env2`)를
Colab 에서는 **1,000 step 만 이어서** 학습합니다.

### (a) 보상 설정 확인 (envs1 / envs2)

이번 라운드에서 비교/사용할 두 보상 설정을 펼쳐 보고, 바뀐 항목을 요약합니다.


In [ ]:
display(show_code(f"{repo_dir}/src/envs1.yaml"))
display(show_code(f"{repo_dir}/src/envs2.yaml"))

**env1 → env2 : 정규화·걸음새 보상 도입**

env1 은 속도 추종(`linear/angular_vel_tracking`)만 보는 *task-only* 설정입니다.
env2 는 여기에 다음을 더해, 마구잡이 동작을 억제하고 안정적인 trot 보행을 유도합니다.

- **정규화 패널티 추가** (모두 0 → 양수): `torque`, `vertical_vel`, `xy_angular_vel`,
  `action_rate`, `joint_limit`, `joint_acc`, `action_norm`, `joint_pos_deviation`
- **걸음새/발**: `gait_enforcement`(trot 패턴 강제), `foot_clearance`(스윙 시 발 높이) 도입
- **종료 조건**: `allow_calf_contact` `true → false` (정강이가 바닥에 닿으면 에피소드 종료)

아래 셀에서 실제로 바뀐 항목과 값을 출력합니다.


In [ ]:
reward_diff(ROUND1_CFG, ROUND2_CFG)   # env1 -> env2

### (b) env2 모델 이어학습 후 확인

`pretrained_env2` 체크포인트를 **`envs2.yaml` 보상으로 1,000 step 이어학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


In [ ]:
r2_model = finetune(ROUND2_MODEL, ROUND2_CFG, run_tag="env2")
vp = rollout_and_video(r2_model, ROUND2_CFG, tag="env2_after")
round_videos.append((vp, "env2 (1,000 step 이어학습 후)"))
show_video(vp)

---

## 5. 라운드 3 — env2 → env3 (positive shaping 추가)

`env3` 보상으로 ~2,000,000 step 미리 학습해 둔 체크포인트(`pretrained_env3`)를
Colab 에서는 1,000 step 만 이어서 학습합니다.

### (a) 보상 설정 확인 (envs2 / envs3)

env2 대비 보상 설정을 펼쳐 보고, 바뀐 항목을 요약합니다.


In [ ]:
display(show_code(f"{repo_dir}/src/envs2.yaml"))
display(show_code(f"{repo_dir}/src/envs3.yaml"))

**env2 → env3 : 자세·생존 보상(positive shaping) 추가**

env2 의 패널티 위주 설정에, 좋은 자세를 직접 장려하는 양(+)의 보상을 더하고
일부 패널티 강도를 미세 조정합니다.

- **positive shaping 추가** (0 → 양수): `healthy`(생존), `base_height`(기준 높이 유지),
  `feet_air_time`(발 체공 시간)
- **패널티 완화**: `vertical_vel`(1.0→0.8), `xy_angular_vel`(0.2→0.15),
  `gait_enforcement`(0.05→0.04), `foot_clearance`(50→25)
- **패널티 강화**: `action_norm`(0.005→0.007), `joint_pos_deviation`(0.05→0.08)

아래 셀에서 실제로 바뀐 항목과 값을 출력합니다.


In [ ]:
reward_diff(ROUND2_CFG, ROUND3_CFG)   # env2 -> env3

### (b) env3 모델 이어학습 후 확인

`pretrained_env3` 체크포인트를 **`envs3.yaml` 보상으로 1,000 step 이어학습**한 뒤,
학습된 모델의 보행을 영상으로 확인합니다.


In [ ]:
r3_model = finetune(ROUND3_MODEL, ROUND3_CFG, run_tag="env3")
vp = rollout_and_video(r3_model, ROUND3_CFG, tag="env3_after")
round_videos.append((vp, "env3 (1,000 step 이어학습 후)"))
show_video(vp)

---

## 6. 결과 비교

env1 / env2 / env3 를 각각 1,000 step 이어학습한 결과 보행을 나란히 비교합니다.


In [ ]:
import base64

def _embed(path, caption, width=320):
    b64 = base64.b64encode(open(path, "rb").read()).decode("ascii")
    return f"""
    <figure style="margin:0; text-align:center;">
      <video controls autoplay muted loop width="{width}">
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
      </video>
      <figcaption style="margin-top:6px; font-size:12px;">{caption}</figcaption>
    </figure>
    """

display(HTML(
    '<div style="display:flex; gap:12px; flex-wrap:wrap;">'
    + "".join(_embed(p, c) for p, c in round_videos if os.path.exists(p))
    + "</div>"
))